<h1 style="text-align: center;">UNIQLO's Global Expansion: Is Bigger Always Better?</h1>

<p style="text-align: center;">
A Financial and Strategic Analysis of Fast Retailing (FY2013–FY2026)
</p>

**Research question:** Has UNIQLO International’s growth become less dependent on expanding its physical store network, and how has this translated into operating profit?

*Scope: UNIQLO Japan and International, FY2013–FY2026. Financial figures are reported in units of 100 million JPY.*

## 1. Data Preparation & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%pip install xlrd openpyxl

In [ ]:
finance_df = pd.read_excel(
    "/Users/jennettehoang/Downloads/20261008_finance2002-2026.xlsx"
)

group_df = pd.read_excel(
    "/Users/jennettehoang/Downloads/20261008_group_gyoseki.xls"
)

In [ ]:
print("FINANCIAL DATA")
display(finance_df.head(11))

print("GROUP PERFORMANCED DATA")
display(group_df.head(11))

In [ ]:
group_clean = group_df.copy()
translations = {
    "国内ユニクロ事業": "UNIQLO Japan",
    "海外ユニクロ事業": "UNIQLO International",
    "売上収益": "Revenue",
    "営業利益": "Operating Profit",
}
mask = group_clean.iloc[:, 1].astype(str).str.contains("期末店舗数")
group_clean.loc[mask, group_clean.columns[1]] = "Number of Stores"
group_clean = group_clean.replace(translations)
display(group_clean.iloc[4:11, :7])

In [ ]:
group_clean = group_clean.rename(columns={
    group_clean.columns[0]: "Region",
    group_clean.columns[1]: "Metric"
})
group_clean["Region"] = group_clean["Region"].ffill()

group_clean = group_clean.loc[4:9].copy()

new_columns = ["Region", "Metric"]

for year in range(2013, 2027):
    new_columns.extend([
        f"FY{year}_Value",
        f"FY{year}_YoY",
        f"FY{year}_Share"
    ])
group_clean.columns = new_columns

group_clean = group_clean.reset_index(drop=True)
display(group_clean.iloc[:, :8])

In [ ]:
id_vars = ["Region", "Metric"]

value_cols = [
    col for col in group_clean.columns
    if col.endswith("_Value")
]

group_long = group_clean.melt(
    id_vars=id_vars,
    value_vars=value_cols,
    var_name="Year",
    value_name="Value"
)

group_long["Year"] = (
    group_long["Year"]
    .str.extract(r"FY(\d{4})")
    .astype(int)
)

display(group_long.head(12))

In [ ]:
uniqlo_df = group_long.pivot(
    index=["Year", "Region"],
    columns="Metric",
    values="Value"
)

uniqlo_df = uniqlo_df.reset_index()

uniqlo_df.columns.name = None

display(uniqlo_df.head(10))

In [ ]:
print(uniqlo_df.dtypes)
print("\nMissing Values:")
print(uniqlo_df.isnull().sum())

In [ ]:
numeric_cols = [
    "Revenue",
    "Operating Profit",
    "Number of Stores"
]
for col in numeric_cols:
    uniqlo_df[col] = pd.to_numeric(
        uniqlo_df[col],
        errors="coerce"
    )
print(uniqlo_df.dtypes)
print("\nMissing Values:")
print(uniqlo_df.isnull().sum())

In [ ]:
print("Dataset Shape:", uniqlo_df.shape)

display(uniqlo_df.tail(10))

In [ ]:
uniqlo_df["Operating Profit Margin"] = (
    uniqlo_df["Operating Profit"] /
    uniqlo_df["Revenue"]
) * 100

display(uniqlo_df.head(11))

In [ ]:
uniqlo_df = uniqlo_df.sort_values(
    by=["Region", "Year"]
)

uniqlo_df["Revenue Growth"] = (
    uniqlo_df.groupby("Region")["Revenue"]
    .pct_change() * 100
)

display(uniqlo_df.head(11))

In [ ]:
uniqlo_df["Store Growth"] = (
    uniqlo_df.groupby("Region")["Number of Stores"]
    .pct_change() * 100
)

uniqlo_df["Revenue per Store"] = (
    uniqlo_df["Revenue"] /
    uniqlo_df["Number of Stores"]
)

display(
    uniqlo_df.loc[
        uniqlo_df["Region"] == "UNIQLO International",
        ["Year", "Revenue Growth", "Store Growth",
         "Revenue per Store", "Operating Profit Margin"]
    ].round(2)
)

## 2. The Rise of UNIQLO International

**Question:** How has the revenue balance between Japan and International changed?

In [ ]:
plt.figure(figsize=(11, 6))

for region in uniqlo_df["Region"].unique():
    data = uniqlo_df[uniqlo_df["Region"] == region]

    plt.plot(
        data["Year"],
        data["Revenue"],
        marker="o",
        linewidth=2,
        label=region
    )

plt.title("UNIQLO: Japan vs International Revenue (FY2013–FY2026)")
plt.xlabel("Fiscal Year")
plt.ylabel("Revenue (100 Million JPY)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(range(2013, 2027))
plt.show()

UNIQLO International surpassed UNIQLO Japan in revenue in FY2018 and has maintained its lead since then. Although both segments experienced revenue declines in FY2020, International recorded a larger contraction. The most notable trend emerged after FY2022, when International revenue grew substantially faster than Japan's, widening the gap between the two segments. This suggests an increasing reliance on international markets as a source of revenue growth.

## 3. Beyond Store Expansion

**Question:** Is international revenue growth being driven mainly by additional stores or changes in revenue per store?

In [ ]:
international = uniqlo_df[
    uniqlo_df["Region"] == "UNIQLO International"
].sort_values("Year")

plt.figure(figsize=(10, 5))

plt.plot(
    international["Year"],
    international["Revenue Growth"],
    marker="o",
    label="Revenue Growth"
)

plt.plot(
    international["Year"],
    international["Store Growth"],
    marker="o",
    label="Store Growth"
)

plt.axhline(0, linestyle="--", color="gray")

plt.title("UNIQLO International: Revenue vs Store Growth")
plt.xlabel("Fiscal Year")
plt.ylabel("Growth Rate (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Revenue growth and store growth followed different trajectories throughout the period. Since FY2022, revenue growth has consistently outpaced store growth, indicating that physical expansion alone cannot fully explain UNIQLO International's recent performance.

This divergence motivates a closer examination of how much revenue growth can be attributed to store expansion versus changes in revenue per store.

### Growth Decomposition: Quantifying the Drivers

The previous analysis revealed a widening gap between revenue growth and store expansion. To quantify this relationship, we decompose annual revenue changes into two components:

- **Store Expansion Effect:** Revenue change associated with changes in store count.
- **Revenue per Store Effect:** Revenue change associated with changes in average revenue per store.

This decomposition allows us to assess how the relative contribution of each component has evolved over time.

In [ ]:
intl = (
    uniqlo_df[uniqlo_df["Region"] == "UNIQLO International"]
    .sort_values("Year")
    .copy()
)

intl["Revenue per Store"] = intl["Revenue"] / intl["Number of Stores"]

intl["Previous Stores"] = intl["Number of Stores"].shift(1)
intl["Previous Revenue per Store"] = intl["Revenue per Store"].shift(1)

intl["Store Expansion Effect"] = (
    (intl["Number of Stores"] - intl["Previous Stores"])
    * intl["Previous Revenue per Store"]
)

intl["Productivity Effect"] = (
    intl["Number of Stores"]
    * (intl["Revenue per Store"] - intl["Previous Revenue per Store"])
)

intl["Revenue Change"] = intl["Revenue"].diff()

decomp = intl.dropna(subset=["Revenue Change"]).copy()


In [ ]:
import numpy as np

years = decomp["Year"].to_numpy()
store = decomp["Store Expansion Effect"].to_numpy()
productivity = decomp["Productivity Effect"].to_numpy()

fig, ax = plt.subplots(figsize=(13, 6))

positive_bottom = np.zeros(len(years))
negative_bottom = np.zeros(len(years))

for values, label, color in [
    (store, "Store Expansion Effect", "#6C8EBF"),
    (productivity, "Revenue per Store Effect", "#E6A157")
]:
    bottoms = np.where(values >= 0, positive_bottom, negative_bottom)

    ax.bar(
        years, values, bottom=bottoms,
        label=label, color=color, width=0.65
    )

    positive_bottom += np.maximum(values, 0)
    negative_bottom += np.minimum(values, 0)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("UNIQLO International: Decomposition of Annual Revenue Change")
ax.set_xlabel("Fiscal Year")
ax.set_ylabel("Revenue Change (100 Million JPY)")
ax.set_xticks(years)
ax.legend()
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### Key Finding: A Shift in International Growth Drivers

The decomposition reveals a shift in UNIQLO International's growth composition. While store expansion contributed substantially in earlier years, changes in revenue per store have dominated annual revenue growth since FY2022.

In FY2026, the revenue per store effect contributed 5,097.6 (100 million JPY), offsetting a negative store expansion effect of 88.6. This suggests that recent growth has become less dependent on increasing the number of physical stores.

**Limitation:** Revenue per store is a proxy rather than a direct measure of store productivity, as reported revenue may include online sales and foreign exchange effects.

## 4. Growth vs. Profitability

**Question:** Has International’s revenue leadership translated into stronger operating profit?

#### Has UNIQLO International's rapid revenue growth translated into higher operating profits compared to Japan?

In [ ]:
plt.figure(figsize=(11, 6))

for region in uniqlo_df["Region"].unique():
    data = uniqlo_df[
        uniqlo_df["Region"] == region
    ].sort_values("Year")

    plt.plot(
        data["Year"],
        data["Operating Profit"],
        marker="o",
        linewidth=2,
        label=region
    )

plt.title("UNIQLO: Japan vs International Operating Profit (FY2013–FY2026)")
plt.xlabel("Fiscal Year")
plt.ylabel("Operating Profit (100 Million JPY)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(range(2013, 2027))
plt.show()

UNIQLO International's operating profit surpassed Japan's in FY2019, but the lead was not sustained during FY2020–FY2021. From FY2022 onward, International experienced a much stronger recovery, with operating profit increasing from 1,583 to 4,481 (100 million JPY) by FY2026, compared with Japan's increase from 1,240 to 1,966.

This widening profit gap suggests that International's growing importance extends beyond revenue contribution, making it an increasingly significant source of operating profit for UNIQLO.

### Additional Context: Revenue Growth and Profitability

UNIQLO International's revenue growth did not consistently translate into higher operating profit margins. In FY2017, for example, operating profitability improved substantially despite relatively modest revenue growth. The FY2020 downturn further highlighted the segment's vulnerability to external disruptions.

These patterns raise an important question: **Does revenue expansion necessarily translate into stronger financial performance?**

## 5. Key Findings & Limitations

- **Scale:** UNIQLO International overtook Japan in revenue and has become an increasingly important contributor to operating profit.
- **Growth composition:** Recent revenue increases are more strongly associated with changes in revenue per store than with changes in store count.
- **Interpretation:** This accounting decomposition does not establish that physical-store productivity caused the increase; online sales, currency translation, pricing and product mix may also matter.

**Strategic implication:** The results motivate investigating the balance between selective new-store expansion and growth through existing channels, rather than assuming more stores always means better performance.